# Chunk → 向量：中间过程（可观测）

串联 **切分 → `TextChunk` → `embed_documents` → L2 / 块间相似度 / 查询相似度**，便于逐步核对输入输出。

**前置**

- **先在项目根执行** `uv sync --extra embedding`（安装 `torch`、`FlagEmbedding` 等）。
- **Jupyter 内核 / 解释器** 必须选本仓库 **`.venv/bin/python`**；若仍报 `No module named 'torch'`，说明内核不是该环境（常见于用了系统 Python 或 conda）。
- 项目根目录 `.env`：已配置 `BGE_M3_PATH` 以及 `MODEL_API_KEY` / `MODEL_BASE_URL` / `MODEL_NAME`（`get_settings()` 会读取）。
- 在 `notebooks/` 下打开本文件时，路径格会把仓库根目录下的 `src` 加入 `PYTHONPATH`。


In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

# 仓库根目录与 PYTHONPATH（笔记本在 notebooks/ 下时）
_ROOT = Path.cwd().resolve()
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
_SRC = _ROOT / "src"
if _SRC.is_dir() and str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

os.chdir(_ROOT)
print("仓库根目录:", _ROOT)
print("已加入 sys.path:", _SRC)


仓库根目录: /Users/zheng/projects/python/ai/rag/rag_law
已加入 sys.path: /Users/zheng/projects/python/ai/rag/rag_law/src


## 0. 检查 `embedding` 运行时（torch / FlagEmbedding）

真实向量化依赖 **PyTorch**。下一格若失败，请在本机项目根执行 `uv sync --extra embedding`，并在 VS Code / Jupyter 里把内核切到 **`.venv`**。


In [3]:
import importlib.util
import sys

print("当前 Python:", sys.executable)

if importlib.util.find_spec("torch") is None:
    raise RuntimeError(
        "未找到 torch。请在项目根目录执行:  uv sync --extra embedding\n"
        "然后将本 notebook 的 Python 内核选为:  <项目根>/.venv/bin/python\n"
        "（当前仍在使用未安装 embedding 的解释器。）"
    )

import torch

try:
    from FlagEmbedding import BGEM3FlagModel  # noqa: F401
except ModuleNotFoundError as e:
    raise RuntimeError(
        "未找到 FlagEmbedding。请执行:  uv sync --extra embedding"
    ) from e

print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
print("embedding 运行时 OK，可继续下面「向量」步骤。")


当前 Python: /Users/zheng/projects/python/ai/rag/rag_law/.venv/bin/python
torch: 2.11.0 | CUDA: False
embedding 运行时 OK，可继续下面「向量」步骤。


## 1. 加载配置（`conf.settings`）

与线上一致：`BGE_M3_PATH`、`EMBEDDING_BATCH_SIZE`、以及默认 `chunk_size` / `chunk_overlap`（仅用于下面演示参数时可覆盖）。


In [4]:
from conf.settings import get_settings

get_settings.cache_clear()
settings = get_settings()

print("BGE_M3_PATH:", settings.bge_m3_path)
print("EMBEDDING_BATCH_SIZE:", settings.embedding_batch_size)
print("settings.chunk_size / chunk_overlap:", settings.chunk_size, settings.chunk_overlap)


BGE_M3_PATH: /Users/zheng/jupyter/c10_edu_rag/rag_qa/models/bge-m3
EMBEDDING_BATCH_SIZE: 32
settings.chunk_size / chunk_overlap: 1500 100


## 2. 准备语料 `full_text`

下面默认使用**短样例**便于阅读；若要对齐黄金测试，可取消注释读取 `tests/test_chunking/fixtures/宪法.md`（块数较多、首次编码较慢）。


In [5]:
# --- 短样例（推荐先跑通）---
SAMPLE_TEXT = """
中华人民共和国宪法（节选样式）

第一条 中华人民共和国是工人阶级领导的、以工农联盟为基础的人民民主专政的社会主义国家。
社会主义制度是中华人民共和国的根本制度。中国共产党领导是中国特色社会主义最本质的特征。

第二条 中华人民共和国的一切权力属于人民。

第五条 中华人民共和国实行依法治国，建设社会主义法治国家。
"""

# --- 可选：与仓库内黄金测试同语料（长）---
# SAMPLE_TEXT = (project_root() / "tests" / "test_chunking" / "fixtures" / "宪法.md").read_text(encoding="utf-8")

full_text = SAMPLE_TEXT.strip()
print("len(full_text) 字符数:", len(full_text))
print("--- 前 500 字预览 ---")
print(full_text[:500])


len(full_text) 字符数: 158
--- 前 500 字预览 ---
中华人民共和国宪法（节选样式）

第一条 中华人民共和国是工人阶级领导的、以工农联盟为基础的人民民主专政的社会主义国家。
社会主义制度是中华人民共和国的根本制度。中国共产党领导是中国特色社会主义最本质的特征。

第二条 中华人民共和国的一切权力属于人民。

第五条 中华人民共和国实行依法治国，建设社会主义法治国家。


## 3. 切分：得到 `list[TextChunk]`

这里显式给出 `chunk_size` / `chunk_overlap`；`boundary_aware=False` 为纯字符滑窗（与句边界版对比时可改为 `True` 并配 floor/ceiling）。

每个 `TextChunk` 含：`text`、`chunk_index`、`char_start`/`char_end`（相对 `full_text` 的切片区间）。


In [6]:
from chunking.split import TextChunk, iter_chunks_for_text

CHUNK_SIZE = 30
CHUNK_OVERLAP = 5

chunks: list[TextChunk] = list(
    iter_chunks_for_text(
        full_text,
        source_file="notebook_sample.md",
        source_path="notebooks/c03_embedding_smoke.ipynb",
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        boundary_aware=False,
    )
)

print(f"块数: {len(chunks)}")
for c in chunks:
    piece = c.text
    preview = piece[:100] + ("…" if len(piece) > 100 else "")
    print(
        f"  [#{c.chunk_index}] char[{c.char_start}:{c.char_end}] len={len(piece)} | 预览: {preview!r}"
    )

# 校验切片与原文一致
for c in chunks:
    assert full_text[c.char_start : c.char_end] == c.text, "TextChunk 与原文切片不一致"
print("校验: 每块 text == full_text[char_start:char_end] OK")


块数: 7
  [#0] char[0:30] len=30 | 预览: '中华人民共和国宪法（节选样式）\n\n第一条 中华人民共和国是工'
  [#1] char[25:55] len=30 | 预览: '共和国是工人阶级领导的、以工农联盟为基础的人民民主专政的社会'
  [#2] char[50:80] len=30 | 预览: '专政的社会主义国家。\n社会主义制度是中华人民共和国的根本制度'
  [#3] char[75:105] len=30 | 预览: '的根本制度。中国共产党领导是中国特色社会主义最本质的特征。\n'
  [#4] char[100:130] len=30 | 预览: '的特征。\n\n第二条 中华人民共和国的一切权力属于人民。\n\n第'
  [#5] char[125:155] len=30 | 预览: '民。\n\n第五条 中华人民共和国实行依法治国，建设社会主义法治'
  [#6] char[150:158] len=8 | 预览: '会主义法治国家。'
校验: 每块 text == full_text[char_start:char_end] OK


## 4. 待编码文本列表（入库 bulk 与这里同构）

`embed_documents` 的输入是 `list[str]`，通常即 `[c.text for c in chunks]`。


In [7]:
texts_to_embed = [c.text for c in chunks]
assert len(texts_to_embed) == len(chunks)

for i, t in enumerate(texts_to_embed):
    print(f"[{i}] 长度={len(t)}")


[0] 长度=30
[1] 长度=30
[2] 长度=30
[3] 长度=30
[4] 长度=30
[5] 长度=30
[6] 长度=8


## 5. BGE-M3：`build_embedder` → `embed_documents`

内部路径：`encode_corpus` → 取 dense → **行 L2 归一化**；`dense_dimension` 在首次成功编码后可用。


In [8]:
from embeddings import build_embedder

embedder = build_embedder(settings)
embeddings: list[list[float]] = embedder.embed_documents(texts_to_embed)

print("dense_dimension:", embedder.dense_dimension)
print("返回向量条数:", len(embeddings), "（应等于块数）")
assert len(embeddings) == len(texts_to_embed)
for i, row in enumerate(embeddings):
    assert len(row) == embedder.dense_dimension, f"行 {i} 维度异常"
print("每行维度与 dense_dimension 一致 OK")


pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 53.53it/s]
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00, 20.09it/s]

dense_dimension: 1024
返回向量条数: 7 （应等于块数）
每行维度与 dense_dimension 一致 OK


## 6. 中间结果：张量形状、L2、前几维、块间余弦

向量已 L2 归一化时，**余弦相似度 = 点积**。


In [9]:
import numpy as np

arr = np.asarray(embeddings, dtype=np.float64)
print("shape (num_chunks, dim) =", arr.shape)

norms = np.linalg.norm(arr, axis=1)
print("每行 L2 范数（应≈1）:", norms)

K = 8
print(f"\n每块向量前 {K} 维:\n", arr[:, :K])

sim = arr @ arr.T
np.set_printoptions(precision=4, suppress=True, linewidth=120)
print("\n块 × 块 余弦相似度矩阵:")
print(sim)


shape (num_chunks, dim) = (7, 1024)
每行 L2 范数（应≈1）: [1. 1. 1. 1. 1. 1. 1.]

每块向量前 8 维:
 [[ 6.08553484e-03  4.64421721e-02 -2.93225254e-02 -8.55098881e-03
   1.11501435e-02 -7.74780940e-04  1.09813349e-02  2.46338937e-02]
 [ 1.51916757e-03 -8.02001179e-03 -4.18377020e-02 -2.13651781e-02
  -2.48539824e-02 -1.00477548e-02  1.65519940e-02 -1.14883456e-02]
 [ 8.16254270e-03  3.55540220e-02 -2.95699767e-02 -2.12620476e-02
  -2.31041986e-02  3.24328587e-05  2.98692689e-02  3.25061972e-02]
 [-2.08260840e-02  4.26273890e-02 -2.69092945e-02 -1.30100748e-02
  -5.00510605e-02 -5.48196808e-02  4.66426819e-02  3.68707421e-02]
 [ 4.76359868e-03  3.07734053e-02 -3.04137189e-02  6.18093825e-03
  -9.24999909e-03 -3.61645896e-02  4.98406665e-02  3.83977432e-03]
 [ 4.32499942e-02  3.78522122e-02 -1.48656441e-02 -5.81160114e-04
   5.13474340e-03  1.88026809e-02  2.48493922e-02  3.16303860e-02]
 [ 1.14949914e-02  2.37286628e-02 -2.62696573e-02 -3.50058183e-02
  -2.33731899e-02  3.59523393e-03  1.94099427e-02

## 7.（可选）查询句：`embed_query` 与各块的相似度

检索时 query 走 `encode_queries`，与 passage 的 `encode_corpus` 分路；此处用点积排序即可观察「哪块最相关」。


In [10]:
query = "国家的根本制度是什么？"
q = np.asarray(embedder.embed_query(query), dtype=np.float64)
print("query 向量 L2:", float(np.linalg.norm(q)), "（应≈1）")

q_sim = arr @ q  # shape (num_chunks,)
order = np.argsort(-q_sim)
print('\n与各块相似度（由高到低）:')
for rank, j in enumerate(order, start=1):
    snippet = texts_to_embed[j][:80].replace('\n', ' ')
    print(f"  {rank}. chunk#{j}  cos={q_sim[j]:.4f}  |  {snippet}…")


Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00,  2.75it/s]

query 向量 L2: 1.0 （应≈1）

与各块相似度（由高到低）:
  1. chunk#2  cos=0.6987  |  专政的社会主义国家。 社会主义制度是中华人民共和国的根本制度…
  2. chunk#3  cos=0.6174  |  的根本制度。中国共产党领导是中国特色社会主义最本质的特征。 …
  3. chunk#1  cos=0.6065  |  共和国是工人阶级领导的、以工农联盟为基础的人民民主专政的社会…
  4. chunk#4  cos=0.5427  |  的特征。  第二条 中华人民共和国的一切权力属于人民。  第…
  5. chunk#6  cos=0.5324  |  会主义法治国家。…
  6. chunk#5  cos=0.4942  |  民。  第五条 中华人民共和国实行依法治国，建设社会主义法治…
  7. chunk#0  cos=0.4823  |  中华人民共和国宪法（节选样式）  第一条 中华人民共和国是工…
